<a href="https://colab.research.google.com/github/solosolve-ai/solosolve-ai/blob/main/manim_video_creator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#installs

In [1]:
# Cell 1: Installation of Manim, ManimML, and dependencies
print("Starting installation... This will take 5-10 minutes.")
!sudo apt-get update -y
!sudo apt-get install -y libcairo2-dev libpango1.0-dev ffmpeg texlive-latex-base texlive-fonts-recommended texlive-fonts-extra texlive-latex-extra
!pip install --upgrade pip setuptools wheel
!pip install manim==0.19.0 manimpango==0.5.0 manim-ml
print("Installation Complete!")

Starting installation... This will take 5-10 minutes.
Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 https://cli.github.com/packages stable InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:7 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to pr

#video

In [17]:
# Cell 1: Installation of Manim, ManimML, and dependencies
# (Assumed to be run successfully from the user's provided code)

# Cell 2: Chapter 2 Scene Definitions (Fully Corrected and Refined)

from manim import *
from scipy.stats import norm
import numpy as np
# Import ManimML
from manim_ml.neural_network import NeuralNetwork, FeedForwardLayer, FeedForwardToFeedForward

# --- SCENE 2.1: The Soul of the Transformer (Attention) ---
# NOTE: This scene is 3D and can be computationally intensive.
class AttentionManifoldScene(ThreeDScene):
    def construct(self):
        title = Text("The Soul of the Transformer: Self-Attention", font_size=36).to_corner(UL).set_z_index(10)
        formula = MathTex(r"\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V", font_size=40).to_corner(UR).set_z_index(10)

        self.add_fixed_in_frame_mobjects(title, formula)
        self.set_camera_orientation(phi=75 * DEGREES, theta=30 * DEGREES, zoom=0.8)

        manifold = Surface(
            lambda u, v: np.array([u, v, 0.2 * np.sin(u) * np.cos(v)]),
            u_range=[-5, 5], v_range=[-5, 5],
            checkerboard_colors=[BLUE_D, BLUE_E], resolution=(24, 24)
        ).scale(1.5)

        self.play(Write(title), Write(formula))
        self.play(Create(manifold))
        self.begin_ambient_camera_rotation(rate=0.08)

        tokens = {
            "shirt": Dot3D(point=manifold.point_from_uv(-1.5, 0.5), color=YELLOW),
            "blue": Dot3D(point=manifold.point_from_uv(0.5, -1.5), color=WHITE),
            "torn": Dot3D(point=manifold.point_from_uv(-2.5, -1.0), color=WHITE)
        }
        # Using fixed_in_frame_mobjects for labels can be tricky with rotation.
        # An alternative is to update their position in each frame. For simplicity, we'll add them as 3D text.
        token_labels = VGroup(*[Text(t, font_size=24).next_to(d, OUT) for t, d in tokens.items()])

        self.play(LaggedStart(*[Create(d) for d in tokens.values()]), Write(token_labels))

        q_vec = Arrow3D(start=tokens["shirt"].get_center(), end=tokens["shirt"].get_center() + np.array([1, 1, 1]), color=BLUE, resolution=8)
        k_vecs = VGroup(*[Arrow3D(start=d.get_center(), end=d.get_center() + np.array([-1, 0.5, 0.5]), color=RED, resolution=8) for d in tokens.values()])
        v_vecs = VGroup(*[Arrow3D(start=d.get_center(), end=d.get_center() + np.array([0, -1, 1]), color=GREEN, resolution=8) for d in tokens.values()])
        self.play(GrowArrow(q_vec), Create(k_vecs), Create(v_vecs))

        # Explicitly show QK^T
        scores_rect = SurroundingRectangle(formula.get_part_by_tex("QK^T"), color=YELLOW)
        self.play(Create(scores_rect))
        self.play(ShowPassingFlash(q_vec.copy().set_color(WHITE), time_width=0.5, run_time=2))
        self.wait(0.5)

        # Show softmax weights
        softmax_rect = SurroundingRectangle(formula.get_part_by_tex("softmax"), color=BLUE)
        self.play(Transform(scores_rect, softmax_rect))

        # Glows represent attention weights, proportional to scores
        glow_blue = Circle(radius=0.4, color=BLUE_A).move_to(tokens["blue"])
        glow_torn = Circle(radius=0.6, color=BLUE_A).move_to(tokens["torn"])
        self.play(FadeIn(glow_blue, glow_torn, scale=2))
        self.wait(1)

        # Weighted sum of V vectors
        weighted_v_blue = v_vecs[1].copy().scale(0.6, scale_tips=True) # Scale V by attention
        weighted_v_torn = v_vecs[2].copy().scale(0.8, scale_tips=True) # Scale V by attention

        final_vec_target = tokens["shirt"].get_center() + weighted_v_blue.get_vector() + weighted_v_torn.get_vector()

        self.play(
            weighted_v_blue.animate.move_to(tokens["shirt"].get_center()),
            weighted_v_torn.animate.move_to(tokens["shirt"].get_center()),
            FadeOut(q_vec, k_vecs, v_vecs[0], v_vecs[1], v_vecs[2]), # Fade out original vectors
            run_time=1.5
        )
        self.play(
            FadeOut(glow_blue, glow_torn, scores_rect, weighted_v_blue, weighted_v_torn),
            tokens["shirt"].animate.move_to(final_vec_target), # Move "shirt" to new context position
            run_time=1.5
        )
        self.wait(2)
        self.stop_ambient_camera_rotation()


# --- SCENE 2.2: Gemma's Architectural Efficiencies (with ManimML) ---

class GemmaEfficiencyScene(Scene):
    def construct(self):
        title = Text("Gemma Efficiency 1: Grouped-Query Attention", font_size=36).to_edge(UP)
        self.play(Write(title))

        queries = VGroup(*[Circle(radius=0.2, color=BLUE) for _ in range(8)]).arrange(RIGHT, buff=0.3).shift(UP*1.5)
        query_text = Text("Queries (Q)", font_size=28).next_to(queries, LEFT)
        keys = VGroup(*[Square(side_length=0.4, color=RED) for _ in range(2)]).arrange(RIGHT, buff=2.2).shift(DOWN*1)
        values = VGroup(*[Triangle(color=GREEN).scale(0.25) for _ in range(2)]).arrange(RIGHT, buff=2).next_to(keys, DOWN, buff=0.2)
        kv_text = Text("Shared K/V", font_size=28).next_to(VGroup(keys, values), LEFT)

        self.play(Write(query_text), Create(queries))
        self.play(Write(kv_text), Create(keys), Create(values))

        group_boxes = VGroup(SurroundingRectangle(queries[0:4]), SurroundingRectangle(queries[4:8]))
        lines = VGroup(*[Arrow(queries[i].get_bottom(), keys[i // 4].get_top(), buff=0.1) for i in range(8)])
        self.play(Create(group_boxes))
        self.play(LaggedStart(*[GrowArrow(line) for line in lines], lag_ratio=0.15)) # Animate the connections
        self.wait(2)
        self.play(FadeOut(queries, query_text, keys, values, kv_text, group_boxes, lines))

        new_title = Text("Gemma Efficiency 2: Interleaved Attention", font_size=36).to_edge(UP)
        self.play(Transform(title, new_title))

        # REFINEMENT: Define distinct layer styles
        layers = [
            FeedForwardLayer(num_nodes=4, node_color=BLUE_C) if (i + 1) % 6 != 0
            else FeedForwardLayer(num_nodes=6, node_color=YELLOW)
            for i in range(12)
        ]

        nn = NeuralNetwork(layers, layer_spacing=0.4).scale(0.8).center()

        # FIX: Access layers by index, not attribute
        local_brace = Brace(VGroup(*nn[0:5]), direction=RIGHT, buff=0.2)
        global_brace = Brace(nn[5], direction=RIGHT, buff=0.2)
        local_label = local_brace.get_text("Local Layers\n(1024 token window)")
        global_label = global_brace.get_text("Global Layer\n(Full Context)")

        self.play(nn.create())
        self.play(GrowFromCenter(local_brace), Write(local_label))
        self.play(GrowFromCenter(global_brace), Write(global_label))
        self.wait(1)
        self.play(nn.make_forward_pass_animation(run_time=4))
        self.wait(2)


# --- SCENE 2.3: The Surgical Tools (QLoRA) ---

class QLoRASurgeryScene(Scene):
    def construct(self):
        title = Text("QLoRA Part 1: Quantization (NF4)", font_size=36).to_edge(UP)
        self.play(Write(title))

        axes = Axes(x_range=[-4, 4, 1], y_range=[0, 0.5, 0.1], x_length=8, y_length=4)
        curve = axes.plot(lambda x: norm.pdf(x), x_range=[-4, 4], color=BLUE)
        self.play(Create(axes), Create(curve))

        # REFINEMENT & FIX: Create non-uniform rectangles for NF4 visualization
        num_quantiles = 16
        boundaries = norm.ppf(np.linspace(1/(2*num_quantiles), 1 - 1/(2*num_quantiles), num_quantiles + 1))

        rects = VGroup()
        for i in range(num_quantiles):
            x1, x2 = boundaries[i], boundaries[i+1]
            # Center of the probability mass in this bucket
            center_x = norm.ppf(( (i+0.5)/num_quantiles + (i+1.5)/num_quantiles ) / 2)
            rect = axes.get_riemann_rectangles(curve, x_range=[x1, x2], dx=(x2-x1), color=interpolate_color(BLUE, GREEN, i/num_quantiles))
            rects.add(rect)

        self.play(LaggedStart(*[Create(rect) for rect in rects], lag_ratio=0.1))
        explanation = Text("Values are chosen based on equal probability areas", font_size=24).to_edge(DOWN)
        self.play(Write(explanation))
        self.wait(2)
        self.play(FadeOut(axes, curve, rects, explanation))

        new_title = Text("QLoRA Part 2: Low-Rank Adaptation", font_size=36).to_edge(UP)
        self.play(Transform(title, new_title))

        formula = MathTex(r"y", r"=", r"\mathbf{W}_0", r"x", r"+", r"\frac{\alpha}{r}", r"\mathbf{B}", r"\mathbf{A}", r"x").scale(1.2)
        self.play(Write(formula))

        w0_box = SurroundingRectangle(formula.get_part_by_tex("W_0"), color=GRAY)
        w0_text = Text("Frozen & Quantized", font_size=24, color=GRAY).next_to(w0_box, DOWN)
        lora_box = SurroundingRectangle(VGroup(formula.get_part_by_tex("B"), formula.get_part_by_tex("A")), color=YELLOW)
        lora_text = Text("Trainable Adapters", font_size=24, color=YELLOW).next_to(lora_box, DOWN)

        self.play(Create(w0_box), Write(w0_text))
        self.play(Create(lora_box), Write(lora_text))
        self.wait(3)


# --- SCENE 2.4: The Full Training Cycle (with ManimML) ---

class TrainingCycleScene(Scene):
    def construct(self):
        title = Text("The Training Cycle", font_size=36).to_edge(UP)
        self.play(Write(title))

        # REFINEMENT: Create a more descriptive network structure
        nn = NeuralNetwork([
            FeedForwardLayer(num_nodes=3, layer_title="Input"),
            FeedForwardLayer(num_nodes=7, layer_title="Gemma (Frozen)", node_color=GRAY),
            FeedForwardLayer(num_nodes=4, layer_title="Heads (Trainable)", node_color=GREEN)
        ], layer_spacing=1.5)
        # Add LoRA adapters visually
        lora_adapters = VGroup(
            Rectangle(height=1.5, width=0.5, color=YELLOW, fill_opacity=0.8).next_to(nn[1], LEFT, buff=0.1),
            Text("LoRA", font_size=20, color=BLACK).move_to(Rectangle(height=1.5, width=0.5).next_to(nn[1], LEFT, buff=0.1))
        )
        nn.scale(0.8).center()
        lora_adapters.scale(0.8).move_to(VGroup(Rectangle(height=1.5, width=0.5).next_to(nn[1], LEFT, buff=0.1).scale(0.8).center()))

        self.play(nn.create())
        self.play(FadeIn(lora_adapters))

        self.play(nn.make_forward_pass_animation(run_time=3))
        self.wait(0.5)

        # FIX: Access layers by index
        gold_labels = Text("Gold Labels", font_size=24).next_to(nn[-1], DOWN, buff=1.5)
        loss_formula = MathTex(r"L = \text{Loss}(\text{Logits}, \text{Labels})", color=RED).next_to(gold_labels, RIGHT, buff=1)
        self.play(Write(gold_labels))
        self.play(
            LaggedStart(
                Arrow(nn[-1].get_bottom(), loss_formula.get_left()),
                Arrow(gold_labels.get_right(), loss_formula.get_left()),
            ), Write(loss_formula)
        )
        self.play(Indicate(loss_formula, color=RED, scale_factor=1.2))

        backprop_title = Text("Backpropagation", font_size=28).to_edge(DOWN)
        self.play(Write(backprop_title))

        # REFINEMENT: Explicitly show which layers are updated
        trainable_layers = VGroup(lora_adapters, nn[-1])
        trainable_highlight = SurroundingRectangle(trainable_layers, color=YELLOW, buff=0.2)
        update_text = Text("Gradients only update trainable weights", font_size=24, color=YELLOW).next_to(trainable_highlight, DOWN)

        self.play(nn.make_backward_pass_animation(run_time=3))
        self.play(Create(trainable_highlight), Write(update_text))
        self.wait(2)

        self.play(FadeOut(backprop_title, trainable_highlight, update_text))

        optimizer_formula = MathTex(r"\theta_{new} = \theta_{old} - \eta \nabla L").to_edge(DOWN)
        self.play(Write(optimizer_formula))

        optimizer_symbol = VGroup(*[
            Rotate(
                Square(side_length=0.2, color=YELLOW),
                angle=PI/4
            ).move_to(p) for p in trainable_layers.get_all_points()[:20] # Sample points
        ])
        self.play(LaggedStart(*[SpinInFromNothing(s) for s in optimizer_symbol]))
        self.wait(2)

In [18]:
# Cell 3: Render the Attention Scene (3D)
%%manim -pql AttentionManifoldScene

Manim Community v0.19.0

[11/02/25 13:22:17] INFO     Animation 0 : Using cached data (hash :                           ]8;id=743234;file:///usr/local/lib/python3.12/dist-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=744878;file:///usr/local/lib/python3.12/dist-packages/manim/renderer/cairo_renderer.py#89\89]8;;\
                             2975571396_2573628411_1651289494)                                                     

[11/02/25 13:22:23] INFO     Animation 1 : Partial movie file written in                   ]8;id=262320;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=874833;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py#588\588]8;;\
                             '/content/media/videos/content/480p15/partial_movie_files/Att                         
                             entionManifoldScene/639311802_2206838921_3503605717.mp4'                              

AttributeError: Surface object has no attribute 'point_from_uv'

In [19]:
# Cell 4: Render the Gemma Efficiency Scene
%%manim -pql GemmaEfficiencyScene

Manim Community v0.19.0

[11/02/25 13:22:39] INFO     Animation 0 : Using cached data (hash :                           ]8;id=632354;file:///usr/local/lib/python3.12/dist-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=490731;file:///usr/local/lib/python3.12/dist-packages/manim/renderer/cairo_renderer.py#89\89]8;;\
                             1185818338_3733031669_223132457)                                                      

                    INFO     Animation 1 : Using cached data (hash :                           ]8;id=703727;file:///usr/local/lib/python3.12/dist-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=753801;file:///usr/local/lib/python3.12/dist-packages/manim/renderer/cairo_renderer.py#89\89]8;;\
                             624642324_3410062998_1874569469)                                                      

[11/02/25 13:22:40] INFO     Animation 2 : Using cached data (hash :                           ]8;id=33339;file:///usr/local/lib/python3.12/dist-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=337083;file:///usr/local/lib/python3.12/dist-packages/manim/renderer/cairo_renderer.py#89\89]8;;\
                             624642324_4008795436_777432167)                                                       

[11/02/25 13:22:41] INFO     Animation 3 : Partial movie file written in                   ]8;id=2879;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=529564;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py#588\588]8;;\
                             '/content/media/videos/content/480p15/partial_movie_files/Gem                         
                             maEfficiencyScene/624642324_3497048396_2881321990.mp4'                                

...

[11/02/25 13:22:42] INFO     Animation 5 : Partial movie file written in                   ]8;id=980190;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=136321;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py#588\588]8;;\
                             '/content/media/videos/content/480p15/partial_movie_files/Gem                         
                             maEfficiencyScene/624642324_2872842549_1481407979.mp4'                                

[11/02/25 13:22:43] INFO     Animation 6 : Partial movie file written in                   ]8;id=974374;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=443148;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py#588\588]8;;\
                             '/content/media/videos/content/480p15/partial_movie_files/Gem                         
                             maEfficiencyScene/624642324_4236123317_3569510680.mp4'                                

[11/02/25 13:22:44] INFO     Animation 7 : Partial movie file written in                   ]8;id=784386;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=777020;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py#588\588]8;;\
                             '/content/media/videos/content/480p15/partial_movie_files/Gem                         
                             maEfficiencyScene/624642324_4194093274_3203057698.mp4'                                

Constructing layers
Current layer: FeedForwardLayer
Current layer: FeedForwardLayer
Current layer: FeedForwardLayer
Current layer: FeedForwardLayer
Current layer: FeedForwardLayer
Current layer: FeedForwardLayer
Current layer: FeedForwardLayer
Current layer: FeedForwardLayer
Current layer: FeedForwardLayer
Current layer: FeedForwardLayer
Current layer: FeedForwardLayer
Current layer: FeedForwardLayer
NeuralNetwork([
    FeedForwardLayer(z_index=3, title_text= , ),
    FeedForwardToFeedForward(input_layer=FeedForwardLayer,output_layer=FeedForwardLayer,)(z_index=2, title_text= , ),
    FeedForwardLayer(z_index=3, title_text= , ),
    FeedForwardToFeedForward(input_layer=FeedForwardLayer,output_layer=FeedForwardLayer,)(z_index=2, title_text= , ),
    FeedForwardLayer(z_index=3, title_text= , ),
    FeedForwardToFeedForward(input_layer=FeedForwardLayer,output_layer=FeedForwardLayer,)(z_index=2, title_text= , ),
    FeedForwardLayer(z_index=3, title_text= , ),
    FeedForwardToFeedForward(i

AttributeError: 'NoneType' object has no attribute 'center'

In [20]:
# Cell 5: Render the QLoRA Scene
%%manim -pql QLoRASurgeryScene

Manim Community v0.19.0

[11/02/25 13:23:03] INFO     Animation 0 : Using cached data (hash :                           ]8;id=539855;file:///usr/local/lib/python3.12/dist-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=608776;file:///usr/local/lib/python3.12/dist-packages/manim/renderer/cairo_renderer.py#89\89]8;;\
                             1185818338_4265206383_223132457)                                                      

[11/02/25 13:23:04] INFO     Animation 1 : Using cached data (hash :                           ]8;id=421206;file:///usr/local/lib/python3.12/dist-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=183804;file:///usr/local/lib/python3.12/dist-packages/manim/renderer/cairo_renderer.py#89\89]8;;\
                             624642324_2499798980_144750668)                                                       

[11/02/25 13:23:05] INFO     Animation 2 : Partial movie file written in                   ]8;id=492849;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=30422;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py#588\588]8;;\
                             '/content/media/videos/content/480p15/partial_movie_files/QLo                         
                             RASurgeryScene/624642324_1157371473_3805062841.mp4'                                   

[11/02/25 13:23:06] INFO     Animation 3 : Partial movie file written in                   ]8;id=457457;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=979020;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py#588\588]8;;\
                             '/content/media/videos/content/480p15/partial_movie_files/QLo                         
                             RASurgeryScene/624642324_2768597617_3218453238.mp4'                                   

...

[11/02/25 13:23:11] INFO     Writing \mathbf{W}_0 to media/Tex/a324def5ae2a1461.tex         ]8;id=199061;file:///usr/local/lib/python3.12/dist-packages/manim/utils/tex_file_writing.py\tex_file_writing.py]8;;\:]8;id=337780;file:///usr/local/lib/python3.12/dist-packages/manim/utils/tex_file_writing.py#111\111]8;;\

                    INFO     Writing + to media/Tex/9f38976a758de16a.tex                    ]8;id=913268;file:///usr/local/lib/python3.12/dist-packages/manim/utils/tex_file_writing.py\tex_file_writing.py]8;;\:]8;id=786417;file:///usr/local/lib/python3.12/dist-packages/manim/utils/tex_file_writing.py#111\111]8;;\

[11/02/25 13:23:12] INFO     Writing \frac{\alpha}{r} to media/Tex/5536014282881041.tex     ]8;id=928327;file:///usr/local/lib/python3.12/dist-packages/manim/utils/tex_file_writing.py\tex_file_writing.py]8;;\:]8;id=119023;file:///usr/local/lib/python3.12/dist-packages/manim/utils/tex_file_writing.py#111\111]8;;\

[11/02/25 13:23:13] INFO     Animation 7 : Partial movie file written in                   ]8;id=461689;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=350790;file:///usr/local/lib/python3.12/dist-packages/manim/scene/scene_file_writer.py#588\588]8;;\
                             '/content/media/videos/content/480p15/partial_movie_files/QLo                         
                             RASurgeryScene/624642324_1766813432_466029929.mp4'                                    

TypeError: Expected all inputs for parameter mobjects to be a Mobjects

In [21]:
# Cell 6: Render the Training Cycle Scene
%%manim -pql TrainingCycleScene

Manim Community v0.19.0

[11/02/25 13:23:24] INFO     Animation 0 : Using cached data (hash :                           ]8;id=258232;file:///usr/local/lib/python3.12/dist-packages/manim/renderer/cairo_renderer.py\cairo_renderer.py]8;;\:]8;id=727537;file:///usr/local/lib/python3.12/dist-packages/manim/renderer/cairo_renderer.py#89\89]8;;\
                             1185818338_1313241714_223132457)                                                      

Constructing layers
Current layer: FeedForwardLayer
Current layer: FeedForwardLayer
Current layer: FeedForwardLayer
NeuralNetwork([
    FeedForwardLayer(z_index=3, title_text= , ),
    FeedForwardToFeedForward(input_layer=FeedForwardLayer,output_layer=FeedForwardLayer,)(z_index=2, title_text= , ),
    FeedForwardLayer(z_index=3, title_text= , ),
    FeedForwardToFeedForward(input_layer=FeedForwardLayer,output_layer=FeedForwardLayer,)(z_index=2, title_text= , ),
    FeedForwardLayer(z_index=3, title_text= , ),
])


AttributeError: 'NoneType' object has no attribute 'center'